**BfPortf 02 Custom target function**

This example shows how to use [bfportf](https://github.com/answering007/bfportf) with a custom metric. [QuantStats](https://github.com/ranaroussi/quantstats) is used for the visual report.

In [ ]:
# Import section
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
pd.set_option('display.precision', 3)

import bfportf as bf
import quantstats as qs


In [ ]:
# Load data
prices = pd.read_csv(
    'https://raw.githubusercontent.com/answering007/bfportf/master/examples/prices.csv',
    index_col=0,
    parse_dates=['Date'],
)
returns = bf.load_returns(prices)
returns.head()


In [ ]:
# Split data to train and test
df_train, df_test = train_test_split(returns, test_size=0.2, shuffle=False)
print(f'Train shape: {df_train.shape}, Test shape: {df_test.shape}')


In [ ]:
# Scale returns
df_train_scaled, train_weights = bf.scale_return_data(df_train)
df_test_scaled, test_weights = bf.scale_return_data(df_test)


In [ ]:
# Custom target function: balance the mean and the standard deviation
def mean_std_param_target(values: np.ndarray, exp: float = 1.0) -> float:
    if exp == 0:
        raise ValueError('exp must be non zero')
    return (1 + values.mean()) ** exp / (1 + values.std()) ** (1 / exp)


In [ ]:
# Run the search with a plain callable metric
found_portfolios = bf.find_portfolios(
    returns=df_train_scaled,
    metric=lambda values: mean_std_param_target(values, exp=3.0),
    custom_weights=train_weights,
    max_number=3,
    verbose=True,
)
results = bf.results_to_frame(found_portfolios)
results.head()


In [ ]:
# Add the usual metrics to the results frame
results['PnL'] = bf.calculate_metric(results, df_train_scaled, train_weights, bf.pnl_target)
results['Sharpe'] = bf.calculate_metric(results, df_train_scaled, train_weights, bf.sharpe_target)
results['Sortino'] = bf.calculate_metric(results, df_train_scaled, train_weights, bf.sortino_target)
results.head()


In [ ]:
# Select the best portfolio
selected_portfolio_symbols = found_portfolios[0].symbols
selected_portfolio_symbols


In [ ]:
# In-sample report
portfolio_series = bf.get_portfolio_return(
    returns=df_train_scaled,
    portfolio_symbols=selected_portfolio_symbols,
    return_weights=train_weights,
)
benchmark_series = bf.get_portfolio_return(
    returns=df_train_scaled,
    portfolio_symbols=['SPY'],
    return_weights=train_weights,
)
qs.reports.full(portfolio_series, benchmark=benchmark_series, title='Portfolio Performance', show=True)


In [ ]:
# Out-of-sample report
portfolio_series = bf.get_portfolio_return(
    returns=df_test_scaled,
    portfolio_symbols=selected_portfolio_symbols,
    return_weights=train_weights,
)
benchmark_series = bf.get_portfolio_return(
    returns=df_test_scaled,
    portfolio_symbols=['SPY'],
    return_weights=train_weights,
)
qs.reports.full(portfolio_series, benchmark=benchmark_series, title='Out-of-sample Performance', show=True)
